In [ ]:
# Imports

import random

import gymnasium as gym
import numpy as np
import torch

from ppo import Actor, Critic

In [ ]:
# Seed

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed) #before constructing the networks because their initial weights are random


In [ ]:
# Environment

env_id = "CartPole-v1"

env = gym.make(env_id)

observation_size = env.observation_space.shape[0]
action_size = env.action_space.n

print("Observation size:", observation_size)
print("Action size:", action_size)

In [ ]:
# Actor and critic

hidden_size = 64

actor = Actor(
    observation_size=observation_size,
    action_size=action_size,
    hidden_size=hidden_size
)

critic = Critic(
    observation_size=observation_size,
    hidden_size=hidden_size
)

print(actor)
print()
print(critic)

In [ ]:
# Reset environment

observation, info = env.reset(seed=seed)

env.action_space.seed(seed)

print("Observation:", observation)
print("Observation shape:", observation.shape)

In [ ]:
# Convert observation to tensor

observation_tensor = torch.as_tensor(
    observation,
    dtype=torch.float32
)

observation_tensor = observation_tensor.unsqueeze(0)

print("Observation tensor:", observation_tensor)
print("Observation tensor shape:", observation_tensor.shape)

In [ ]:
# Get actor logits

with torch.no_grad():

    logits = actor(observation_tensor)

print("Logits:", logits)
print("Logits shape:", logits.shape)

In [ ]:
# Create action distribution

action_distribution = torch.distributions.Categorical(
    logits=logits
)

print("Action probabilities:", action_distribution.probs)
print("Probability sum:", action_distribution.probs.sum(dim=-1))

In [ ]:
# Sample action

with torch.no_grad():

    action_tensor = action_distribution.sample()

print("Action tensor:", action_tensor)
print("Action tensor shape:", action_tensor.shape)

In [ ]:
# Calculate old log-probability

with torch.no_grad():

    old_log_prob = action_distribution.log_prob(
        action_tensor
    )

print("Old log-probability:", old_log_prob)
print("Old log-probability shape:", old_log_prob.shape)

In [ ]:
# Estimate state value

with torch.no_grad():

    value = critic(observation_tensor)

print("Value:", value)
print("Value shape:", value.shape)

In [ ]:
# Check shapes

assert logits.shape == (1, action_size)
assert action_tensor.shape == (1,)
assert old_log_prob.shape == (1,)
assert value.shape == (1,)

print("Actor and critic checks passed.")

# These are worthwhile because they protect us against exactly the kind of silent tensor-shape errors that become painful once PPO is running.

In [ ]:
# Take one environment step

action = action_tensor.item()

next_observation, reward, terminated, truncated, info = env.step(
    action
)

print("Action:", action)
print("Next observation:", next_observation)
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)

In [ ]:
# Close environment

env.close()